# migrate to silver schema

In [0]:
from pyspark.sql import functions as F

# Read the corrected Bronze table
df = spark.read.table("flightdata.bronze.flightData")

# Sanity check — confirm types look right before merging
df.printSchema()
df.show(2, truncate=False)

# Merge year/month/day into a single date column
df_silver = df.withColumn(
    "date",
    F.to_date(
        F.concat_ws("-", F.col("year"), F.col("month"), F.col("day")),
        "yyyy-M-d"
    )
).drop("year", "month", "day")

# Write to Silver as a managed Delta table
df_silver.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("flightdata.silver.flightdata")

# Verify
display(spark.read.table("flightdata.silver.flightdata"))

In [0]:
%sql
select * from flightdata.silver.flightdata

## check for duplicates.

In [0]:
df_silver = spark.read.table("flightdata.silver.flightdata")

total_count = df_silver.count()
distinct_count = df_silver.distinct().count()

print(f"Total rows: {total_count}")
print(f"Distinct rows: {distinct_count}")
print(f"Duplicate rows: {total_count - distinct_count}")

from pyspark.sql import functions as F

dup_rows = df_silver.groupBy(df_silver.columns).count().filter(F.col("count") > 1)
display(dup_rows)

## Check for nulls

In [0]:
from pyspark.sql import functions as F

df_silver = spark.read.table("flightdata.silver.flightdata")

# Count nulls per column
null_counts = df_silver.select(
    [F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df_silver.columns]
)

display(null_counts)